# Generalised Psychophysiological Interaction (gPPI) Tutorial

**Goal:** Compute a parcel-level **ROI-to-ROI** gPPI connectivity matrix from fMRIPrep preprocessed *task* data.

Every parcel acts as a seed in turn, producing a full N × N matrix of task-modulated connectivity — no single seed needs to be designated in advance.

```
fMRIPrep BOLD  ──►  confound regression  ──►  parcellation  ──►  HRF regressors  ──►  gPPI GLM  ──►  connectivity matrix  ──►  CSV
```

---

### How gPPI differs from standard FC

| | Standard FC | gPPI |
|--|-------------|------|
| **Data** | Resting-state or task | Task BOLD with event timing |
| **Method** | Pearson correlation | OLS regression with PPI interaction terms |
| **Output** | Correlation matrix (Fisher z) | Beta-weight matrix (OLS coefficients) |
| **Question** | "Which regions co-vary?" | "Which regions co-vary *specifically during a task condition*?" |
| **Bandpass** | 0.01–0.10 Hz | High-pass only (keep task-frequency power) |
| **Standardize** | Yes (z-score) | No (preserves beta scale) |

---

**How this notebook is structured**

| Section | What to do |
|---------|------------|
| 1. Imports | Run as-is |
| 2. Configuration | Fill in your paths, task label, and conditions |
| 3. Explore the helpers | Run `fc_help()` to see available functions |
| 4 – 13. Analysis | Each cell has a `# YOUR CODE HERE` block to complete |
| 14. Answer key | Scroll down for a complete working solution |

---
*Dependencies: see `README.md` for conda environment setup.*  
*Helper functions: see `fc_helpers.py` — call `fc_help("function_name")` for detailed docs.*

## 1. Imports

In [4]:
import os
import warnings
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import nibabel as nib
from nilearn import plotting
from pathlib import Path

# Import all helper functions + the help browser
from fc_helpers import *

warnings.filterwarnings('ignore')
%matplotlib inline
print('Imports OK')

# nice for testing / debugging helper functions
import importlib, fc_helpers
importlib.reload(fc_helpers)
from fc_helpers import *

# Confirm helper location
print(fc_helpers.__file__)

Imports OK
/pl/active/courses/2026_summer/neuroclass2026/software/tutorials/fc_helpers.py


## 2. Configuration

Set every path and parameter here. Nothing downstream should need editing.

> **gPPI-specific notes**
> - `TASK` should be a task run, **not** `'rest'`.
> - `TASK_CONDITIONS` lists the `trial_type` values you want PPI terms for.  Leave as `None` to model all conditions.
> - `SEED_LABEL` is the ROI used for the per-seed design-matrix inspection step.  Any valid label from the parcellation works.
> - **Do not** apply low-pass filtering (`LOWPASS_HZ = None`) — task responses contain power above 0.10 Hz.
> - **Do not** Z-score the time series (`STANDARDIZE = False`) — gPPI betas must be in signal units to be interpretable.

In [5]:
# # ── Subject / run identifiers ─────────────────────────────────────────────────
# SUBJECT  = 'sub-MSC01'
# SESSION  = 'ses-func01'
# TASK     = 'glasslexical'   # ← task label (NOT 'rest')
# RUN      = None             # set to 'run-01' if your data has run-level files
# SPACE    = 'MNI152NLin2009cAsym'

# # ── fMRIPrep derivatives root ─────────────────────────────────────────────────
# FMRIPREP_DIR = '/path/to/fmriprep'

FMRIPREP_DIR = '/path/to/fmriprep'
SUBJECT      = 'sub-XXXX'
SESSION      = 'ses-XXXX'   # set to None if no session level
TASK         = 'motor'
RUN          = 'run-01'   # set to None if no run level
SPACE        = 'MNI152NLin6Asym'

# ── gPPI-specific settings ────────────────────────────────────────────────────
# Leave as None to model ALL trial_type values in the events file.
# Or pass a list, e.g. ['face', 'scene'], to model only those conditions.
TASK_CONDITIONS = None

# ── Build file paths automatically from the identifiers above ────────────────
_paths         = build_fmriprep_paths(
    fmriprep_dir = FMRIPREP_DIR,
    subject      = SUBJECT,
    session      = SESSION,
    task         = TASK,
    run          = RUN,
    space        = SPACE,
)
BOLD_PATH      = _paths['BOLD_PATH']
CONFOUNDS_PATH = _paths['CONFOUNDS_PATH']
EVENTS_PATH    = _paths['EVENTS_PATH']   # required for gPPI

# ── Parcellation ──────────────────────────────────────────────────────────────
ATLAS        = 'schaefer'  # 'schaefer' | 'destrieux' | 'custom'
N_ROIS       = 200
YEO_NETWORKS = 7
ATLAS_PATH   = None        # '/path/to/my_atlas.nii.gz'

# ── Acquisition parameters ────────────────────────────────────────────────────
TR = 2.0   # repetition time in seconds

# ── Confound strategy ─────────────────────────────────────────────────────────
# Same 24-parameter motion model as the FC tutorial.
# Global signal regression is OFF for task data — the task signal shares
# variance with the global signal, so GSR would remove activation of interest.
CONFOUND_COLS = [
    'trans_x', 'trans_y', 'trans_z', 'rot_x', 'rot_y', 'rot_z',
    'trans_x_derivative1', 'trans_y_derivative1', 'trans_z_derivative1',
    'rot_x_derivative1',   'rot_y_derivative1',   'rot_z_derivative1',
    'trans_x_power2',      'trans_y_power2',       'trans_z_power2',
    'rot_x_power2',        'rot_y_power2',         'rot_z_power2',
    'trans_x_derivative1_power2', 'trans_y_derivative1_power2',
    'trans_z_derivative1_power2', 'rot_x_derivative1_power2',
    'rot_y_derivative1_power2',   'rot_z_derivative1_power2',
    'white_matter', 'csf',
    # 'global_signal' intentionally excluded for task data
]
SCRUB_THRESHOLD = 0.5   # FD in mm; set to None to skip scrubbing


# ── Time-series extraction settings for gPPI ──────────────────────────────────
# Low-pass is disabled so task-frequency signal is preserved.
# Standardize is False so OLS betas remain in BOLD-signal units.
LOWPASS_HZ  = None    # ← None (not 0.1) for task / gPPI
HIGHPASS_HZ = 0.01
STANDARDIZE = False   # ← False for gPPI (True for standard FC)

# ── Output ────────────────────────────────────────────────────────────────────
OUTPUT_DIR = os.getenv('OUTPUT_DIR', str(Path(FMRIPREP_DIR).parent / 'gppi_matrices'))
os.makedirs(OUTPUT_DIR, exist_ok=True)

# ── Output filename ───────────────────────────────────────────────────────────
_sess_tag       = f'_{SESSION}' if SESSION else ''
OUTPUT_FILENAME = (f'{SUBJECT}{_sess_tag}_task-{TASK}'
                   f'_atlas-{ATLAS}{N_ROIS}_gppi.csv')

print('Configuration ready.')
print(f'  BOLD      : {BOLD_PATH}')
print(f'  Confounds : {CONFOUNDS_PATH}')
print(f'  Events    : {EVENTS_PATH}')
print(f'  Atlas     : {ATLAS} | N_ROIS={N_ROIS}')
print(f'  Low-pass  : {LOWPASS_HZ}  (None = task-frequency signal preserved)')
print(f'  Standardize: {STANDARDIZE}  (False = betas in signal units)')
print(f'  Output    : {OUTPUT_DIR}')

# ── Pipeline logger ───────────────────────────────────────────────────────────
init_log({
    'SUBJECT'          : SUBJECT,
    'SESSION'          : SESSION,
    'TASK'             : TASK,
    'RUN'              : RUN,
    'SPACE'            : SPACE,
    'BOLD_PATH'        : BOLD_PATH,
    'CONFOUNDS_PATH'   : CONFOUNDS_PATH,
    'EVENTS_PATH'      : EVENTS_PATH,
    'ATLAS'            : ATLAS,
    'N_ROIS'           : N_ROIS,
    'YEO_NETWORKS'     : YEO_NETWORKS,
    'ATLAS_PATH'       : ATLAS_PATH,
    'TR'               : TR,
    'SCRUB_THRESHOLD'  : SCRUB_THRESHOLD,
    'TASK_CONDITIONS'  : TASK_CONDITIONS,
    'LOWPASS_HZ'       : LOWPASS_HZ,
    'HIGHPASS_HZ'      : HIGHPASS_HZ,
    'STANDARDIZE'      : STANDARDIZE,
    'CONFOUND_COLS'    : CONFOUND_COLS,
    'OUTPUT_DIR'       : OUTPUT_DIR,
    'OUTPUT_FILENAME'  : OUTPUT_FILENAME,
})

  [paths] BOLD        : /scratch/alpine/amhe4269/openneuro/ds000224/derivatives/fmriprep/sub-MSC02/ses-func01/func/sub-MSC02_ses-func01_task-motor_run-01_space-MNI152NLin6Asym_res-2_desc-preproc_bold.nii.gz
  [paths] Confounds   : NOT FOUND ⚠
  [paths] Events      : /scratch/alpine/amhe4269/openneuro/ds000224/derivatives/fmriprep/sub-MSC02/ses-func01/func/sub-MSC02_ses-func01_task-motor_run-01_events.tsv
Configuration ready.
  BOLD      : /scratch/alpine/amhe4269/openneuro/ds000224/derivatives/fmriprep/sub-MSC02/ses-func01/func/sub-MSC02_ses-func01_task-motor_run-01_space-MNI152NLin6Asym_res-2_desc-preproc_bold.nii.gz
  Confounds : None
  Events    : /scratch/alpine/amhe4269/openneuro/ds000224/derivatives/fmriprep/sub-MSC02/ses-func01/func/sub-MSC02_ses-func01_task-motor_run-01_events.tsv
  Atlas     : schaefer | N_ROIS=200
  Low-pass  : None  (None = task-frequency signal preserved)
  Standardize: False  (False = betas in signal units)
  Output    : /scratch/alpine/amhe4269/openneuro/

## 3. Explore the Helper Functions

Run the cells below to see what's available. Each helper has detailed documentation.

In [ ]:
# List all available helper functions
fc_help()

In [ ]:
# Read detailed docs for any gPPI function before you use it, e.g.:
fc_help('load_events')

In [ ]:
# Try a few others:
# fc_help('build_hrf_regressors')
# fc_help('build_gppi_design')
# fc_help('compute_gppi_matrix')
# fc_help('plot_gppi_seed_profile')
# fc_help('plot_hrf_regressors')
# fc_help('plot_gppi_design_matrix')
# fc_help('summarise_gppi')

---
## 4. Inspect Motion Quality

Before any analysis, check the framewise displacement (FD) trace to understand data quality.

**Function:** `plot_fd_trace(confounds_path, scrub_threshold, t_r)`  
**Returns:** `fig, mean_fd, pct_scrubbed`

> For task fMRI, Power et al. (2012) suggest a slightly more permissive threshold (0.9 mm) than resting-state (0.5 mm), because the task signal itself induces small head movements at stimulus onsets.

👉 *Call `fc_help('plot_fd_trace')` for full parameter documentation.*

In [ ]:
# ── YOUR CODE HERE ────────────────────────────────────────────────────────────
# Plot the FD trace using plot_fd_trace().
# Pass: CONFOUNDS_PATH, SCRUB_THRESHOLD, and TR
# Capture the three return values: fig, mean_fd, pct_scrubbed



# ─────────────────────────────────────────────────────────────────────────────
plt.show()
# After running: is mean FD < 0.5 mm? What percentage of volumes were scrubbed?

## 5. Load Confound Regressors

**Function:** `load_confounds(confounds_path, cols, scrub_threshold)`  
**Returns:** `confounds_array` (shape T × regressors), `scrubbed_idx`

The `CONFOUND_COLS` list is already built in the Configuration cell with global signal **excluded** — the correct choice for task fMRI.

> **Why no GSR for task data?**  
> The global signal partially reflects widespread neural activation.  During a task, this shared activation is signal, not noise.  Regressing it out would remove the very variance you want to model in the gPPI.

👉 *Call `fc_help('load_confounds')` to see full confound strategy options.*

In [ ]:
# ── YOUR CODE HERE ────────────────────────────────────────────────────────────
# Load the confound regressors using load_confounds().
# Pass: CONFOUNDS_PATH, CONFOUND_COLS, SCRUB_THRESHOLD
# Store results as: confounds_array, scrubbed_idx



# ─────────────────────────────────────────────────────────────────────────────
# After running: how many regressors? Are any columns missing from your file?

## 6. Load the Parcellation

**Function:** `get_parcellation(atlas, n_rois, yeo_networks, atlas_path)`  
**Returns:** `atlas_img` (NIfTI), `roi_labels` (list of str)

👉 *Call `fc_help('get_parcellation')` for atlas options.*

In [ ]:
# ── YOUR CODE HERE ────────────────────────────────────────────────────────────
# Load the parcellation using get_parcellation().
# Use the ATLAS, N_ROIS, YEO_NETWORKS, and ATLAS_PATH variables from CONFIG.
# Store results as: atlas_img, roi_labels



# ─────────────────────────────────────────────────────────────────────────────
# Visualise the parcellation on a brain
plotting.plot_roi(
    atlas_img,
    title        = f'{ATLAS.capitalize()}-{N_ROIS} parcellation',
    display_mode = 'z',
    cut_coords   = 6,
    colorbar     = True,
)
plt.show()

print(f'Number of ROI labels: {len(roi_labels)}')
print(f'First 5 labels      : {roi_labels[:5]}')


🛑 **check that your parcellation map and fMRI time series data are in the same space!** Are both aligned to the same anatomical template? Are they both in the same resolution (e.g. 2 mm)? Do not proceed unless you are confident your mask and input data are aligned correctly!

## 7. Check BOLD ↔ Atlas Alignment

**Functions:**
- `check_alignment(bold_path, atlas_img, t_r)` — prints a five-point numeric report
- `plot_alignment(bold_path, atlas_img)` — overlays atlas borders on the mean BOLD

👉 *Call `fc_help('check_alignment')` or `fc_help('plot_alignment')` for full docs.*

In [ ]:
# ── YOUR CODE HERE ────────────────────────────────────────────────────────────
# 7a. Run the numeric alignment check using check_alignment().
#     Pass: BOLD_PATH, atlas_img, and TR
#     Store the result as: report



# ─────────────────────────────────────────────────────────────────────────────
# If space_label FAILED — stop here and fix the template space mismatch.

In [ ]:
# ── YOUR CODE HERE ────────────────────────────────────────────────────────────
# 7b. Visually verify alignment using plot_alignment().
#     Pass: BOLD_PATH and atlas_img
#     Store the result as: display



# ─────────────────────────────────────────────────────────────────────────────
# What to look for: atlas parcel borders should follow gyral anatomy.
# display.savefig('./gppi_output/alignment_check.png', dpi=150)

## 8. Extract Task Time Series

**Function:** `extract_time_series(bold_path, atlas_img, confounds_array, t_r, ...)`  
**Returns:** `time_series` (shape T × n_rois)

⚠️ **Two settings differ from the standard FC pipeline:**

| Setting | FC notebook | gPPI notebook | Why |
|---------|------------|---------------|-----|
| `standardize` | `True` | **`False`** | gPPI betas must be in BOLD-signal units so regression coefficients are interpretable |
| `low_pass` | `0.10 Hz` | **`None`** | Task-evoked responses contain power above 0.10 Hz; filtering them out removes the very signal you want to model |

👉 *Call `fc_help('extract_time_series')` for all parameter details.*

In [ ]:
# ── YOUR CODE HERE ────────────────────────────────────────────────────────────
# Extract parcel time series using extract_time_series().
# Pass: BOLD_PATH, atlas_img, confounds_array, TR
# IMPORTANT: use standardize=STANDARDIZE (False) and low_pass=LOWPASS_HZ (None)
# Store result as: time_series



# ─────────────────────────────────────────────────────────────────────────────
# Visualise a sample of extracted time series
fig = plot_time_series(time_series, TR, n_rois=8, labels=roi_labels)
plt.show()
# After running: does the shape match your expectations? (T × N_ROIS)

## 9. Load Task Events

**Function:** `load_events(events_path, conditions)`  
**Returns:** `events_df` (DataFrame with onset, duration, trial_type), `conditions_found`

The events file is the BIDS `_events.tsv` located alongside your BOLD file.  It tells the gPPI model *when* each condition occurred, so the HRF regressors can be built correctly.

> If `EVENTS_PATH` is `None` from the Configuration cell, fMRIPrep did not find an events file for your run.  Check that the file exists with the correct BIDS naming convention.

👉 *Call `fc_help('load_events')` for full parameter documentation.*

In [ ]:
# ── YOUR CODE HERE ────────────────────────────────────────────────────────────
# Load task events using load_events().
# Pass: EVENTS_PATH and TASK_CONDITIONS
# Store results as: events_df, conditions_found
#
# Tip: to model specific conditions, set e.g.:
#   TASK_CONDITIONS = ['checkerboard_left', 'checkerboard_right']
# Leave as None to model every trial_type found in the events file.



# ─────────────────────────────────────────────────────────────────────────────
# If TASK_CONDITIONS was None, pin it to whatever was found so that
# subsequent cells can iterate over it with enumerate():
# if TASK_CONDITIONS is None:
#     TASK_CONDITIONS = conditions_found
#
# Inspect the events:
# print(events_df.head(10))
# events_df['trial_type'].value_counts()


## 10. Build HRF Regressors

**Function:** `build_hrf_regressors(events_df, t_r, n_scans, conditions, hrf_model)`  
**Returns:** `hrf_regressors` (dict: condition → array of shape `(n_scans,)`)

This step convolves your event timing with the canonical haemodynamic response function (HRF) to produce a predicted BOLD signal for each condition.  These become the **psychological regressors** in the gPPI model.

```
event boxcar  ──►  ⊗ HRF  ──►  predicted BOLD trace
```

👉 *Call `fc_help('build_hrf_regressors')` for full parameter documentation.*

In [ ]:
# ── YOUR CODE HERE ────────────────────────────────────────────────────────────
# Build HRF-convolved regressors using build_hrf_regressors().
# Pass: events_df, TR, n_scans=time_series.shape[0]
# Store result as: hrf_regressors



# ─────────────────────────────────────────────────────────────────────────────
# Visualise the regressors using plot_hrf_regressors()



# ─────────────────────────────────────────────────────────────────────────────
plt.show()
# After running: do the peaks align with when you expect neural activity?

## 11. Inspect One ROI's gPPI Design Matrix

**Function:** `build_gppi_design(seed_ts, hrf_regressors, confounds)`  
**Returns:** `design_matrix` (T × n_columns), `col_names`, `ppi_cols`

Before running the full ROI-to-ROI pipeline, inspect the design matrix for **one example ROI** to verify that the four column blocks are correctly assembled.  In the full computation (`compute_gppi_matrix`) this design is built and solved for **every** ROI in turn — what you see here is one representative instance of that N-times-repeated process.

Pick any ROI index you like for this inspection step.  `roi_idx = 0` is the simplest choice; you can change it to any value in `range(N_ROIS)`.

**Design matrix structure:**

| Block | Columns | Description |
|-------|---------|-------------|
| Physiological | `seed` | Mean BOLD of the current ROI (changes each iteration) |
| Psychological | `psych_<condition>` | HRF-convolved task regressor (same for every ROI) |
| PPI interaction | `ppi_<condition>` | ROI ts × psychological (the connectivity term of interest) |
| Nuisance | `conf_000`, `conf_001`, … | Confound regressors + intercept |

👉 *Call `fc_help('build_gppi_design')` for full parameter documentation.*


In [ ]:
# ── YOUR CODE HERE ────────────────────────────────────────────────────────────
# Build the design matrix for one example ROI using build_gppi_design().
#
# Step 1: choose any ROI index (0 = first ROI, or any value in range(N_ROIS))
#   roi_idx = 0
# Step 2: extract that ROI's time series (shape T,)
#   roi_ts = time_series[:, roi_idx]
# Step 3: call build_gppi_design()
#   Pass: roi_ts, hrf_regressors, confounds=confounds_array
#   Store results as: design_matrix, col_names, ppi_cols



# ─────────────────────────────────────────────────────────────────────────────
print(f'Design matrix shape : {design_matrix.shape}  (T × regressors)')
print(f'Column names        : {col_names}')
print(f'PPI column indices  : {ppi_cols}')
# In the full ROI-to-ROI computation, this design is rebuilt for each of the
# N_ROIS ROIs, with a different physiological column each time.


In [ ]:
# ── YOUR CODE HERE ────────────────────────────────────────────────────────────
# Visualise the design matrix using plot_gppi_design_matrix().
# Pass: design_matrix, col_names
# Add a descriptive title that includes SUBJECT and the ROI label.
#   Tip: roi_labels[roi_idx]



# ─────────────────────────────────────────────────────────────────────────────
plt.show()
# After running: can you see the four colour-coded blocks?
# The PPI column should look like the product of the seed and psychological columns.


## 12. Compute the gPPI Connectivity Matrix

**Function:** `compute_gppi_matrix(time_series, events_df, t_r, confounds, conditions, ...)`  
**Returns:** `gppi_matrix` — shape **(n_rois × n_rois × n_conditions)**  
Access a condition slice with `gppi_matrix[:, :, idx]`.

This is the core step.  For each seed ROI it:
1. Extracts the seed time series
2. Assembles the gPPI design matrix
3. Fits OLS against all target ROIs simultaneously
4. Stores each condition's PPI beta along axis-2 of the output array

Because every ROI acts as a seed in turn, the result is a **complete N × N matrix**
for each condition, allowing condition-specific connectivity contrasts
(e.g., `gppi_matrix[:,:,0] - gppi_matrix[:,:,1]`).


In [ ]:
# ── YOUR CODE HERE ────────────────────────────────────────────────────────────
# Compute the gPPI matrix using compute_gppi_matrix().
# Pass: time_series, events_df, TR, confounds=confounds_array
# Pass: conditions=TASK_CONDITIONS
# Returns a 3D array (n_rois, n_rois, n_conditions).
# Store result as: gppi_matrix



# ─────────────────────────────────────────────────────────────────────────────
# Inspect: print(f'gPPI matrix shape: {gppi_matrix.shape}')
# Summarise each condition slice:
# for idx, cond in enumerate(TASK_CONDITIONS):
#     summarise_gppi(gppi_matrix[:,:,idx], label=f'{SUBJECT} | {cond}')


## 13. Visualise and Save

**Functions:** `plot_fc_matrix(...)`, `plot_gppi_seed_profile(...)`, and `save_fc_matrix(...)`

> `plot_fc_matrix` and `save_fc_matrix` work identically for gPPI matrices — just change the title/filename to indicate the data are gPPI betas, not Pearson r.

After computing the full ROI-to-ROI matrix you can inspect any ROI's connection profile.  Pick a `PROFILE_ROI` label of interest from `roi_labels` (e.g. `roi_labels[0]`, or any label you know is relevant to your task) and pass it to `plot_gppi_seed_profile()`.

👉 *Call `fc_help('plot_gppi_seed_profile')` for seed-profile options.*


In [ ]:
# ── YOUR CODE HERE ────────────────────────────────────────────────────────────
# 13a. Plot the gPPI heatmap for each condition.
#      Loop over conditions, slicing gppi_matrix[:,:,idx] for each.
#      Use plot_fc_matrix() with labels=roi_labels and a descriptive title
#      that includes the condition name.
#      Tip: the colour scale represents gPPI beta weights (not Fisher z).



# ─────────────────────────────────────────────────────────────────────────────
plt.show()


In [ ]:
# ── YOUR CODE HERE ────────────────────────────────────────────────────────────
# 13b. Choose any ROI and inspect its connection profile per condition.
#
# Step 1: pick a ROI
#   PROFILE_ROI = roi_labels[0]   # or any label from roi_labels
# Step 2: loop over conditions and call plot_gppi_seed_profile()
#   for idx, COND in enumerate(TASK_CONDITIONS):
#       plot_gppi_seed_profile(gppi_matrix[:,:,idx], labels=roi_labels,
#                              seed_label=PROFILE_ROI, n_top=20)



# ─────────────────────────────────────────────────────────────────────────────
plt.show()


In [ ]:
# ── YOUR CODE HERE ────────────────────────────────────────────────────────────
# 13c. Save each condition's matrix using save_fc_matrix().
#      Loop over conditions with enumerate(TASK_CONDITIONS),
#      slice gppi_matrix[:,:,idx], and build a filename that includes
#      SUBJECT, TASK, atlas info, and the condition name.
#      Example filename: f'{SUBJECT}_task-{TASK}_atlas-{ATLAS}{N_ROIS}_cond-{COND}_gppi.csv'



# ─────────────────────────────────────────────────────────────────────────────
# Verify the last saved file:
# check = pd.read_csv(out_path, index_col=0)
# print(f'Verified CSV shape: {check.shape}')   # should be (N_ROIS, N_ROIS)


---
## 🔬 Checkpoint — Questions to Consider

Before moving to the answer key, think through these:

1. **Design matrix** — What are the four column blocks in a gPPI design matrix, and what does each one control for?
2. **HRF convolution** — Why do we convolve the event boxcar with an HRF before computing the PPI interaction term?
3. **Low-pass filtering** — Why is `low_pass=None` required for gPPI but not for resting-state FC?
4. **Standardize=False** — If you standardised the time series (z-scored) before computing gPPI, how would that affect your ability to interpret the beta coefficients?
5. **Asymmetry** — Why can gPPI matrices be asymmetric (i.e. `matrix[i,j] ≠ matrix[j,i]`), and when would you symmetrise before group analysis?
6. **gPPI vs FC** — A region shows high FC with the seed at rest but near-zero gPPI during the task.  What does this imply about its relationship with the seed?

---

# ✅ Answer Key

---
> **Scroll past here only after completing the exercise above.**
> 
> The cells below show a complete, working solution for each step.
---

### AK-1. Motion Quality

In [ ]:
# ── ANSWER KEY: Step 4 — Inspect FD trace ────────────────────────────────────
fig, mean_fd, pct_scrubbed = plot_fd_trace(
    confounds_path  = CONFOUNDS_PATH,
    scrub_threshold = SCRUB_THRESHOLD,
    t_r             = TR,
)
plt.show()
print(f'Mean FD: {mean_fd:.3f} mm  |  {pct_scrubbed:.1f}% of volumes scrubbed')

### AK-2. Load Confounds

In [ ]:
# ── ANSWER KEY: Step 5 — Load confound regressors ────────────────────────────
# Global signal is excluded from CONFOUND_COLS — correct for task fMRI.
confounds_array, scrubbed_idx = load_confounds(
    confounds_path  = CONFOUNDS_PATH,
    cols            = CONFOUND_COLS,
    scrub_threshold = SCRUB_THRESHOLD,
)
print(f'Confound matrix : {confounds_array.shape}  (T x regressors)')
print(f'Scrubbed volumes: {len(scrubbed_idx)}')

### AK-3. Load Parcellation

In [ ]:
# ── ANSWER KEY: Step 6 — Load parcellation ───────────────────────────────────
atlas_img, roi_labels = get_parcellation(
    atlas        = ATLAS,
    n_rois       = N_ROIS,
    yeo_networks = YEO_NETWORKS,
    atlas_path   = ATLAS_PATH,
)
print(f'Atlas image shape   : {atlas_img.shape}')
print(f'Number of ROI labels: {len(roi_labels)}')

plotting.plot_roi(
    atlas_img,
    title        = f'{ATLAS.capitalize()}-{N_ROIS} parcellation',
    display_mode = 'z',
    cut_coords   = 6,
    colorbar     = True,
)
plt.show()

### AK-4. Alignment QC

In [ ]:
# ── ANSWER KEY: Step 7 — Check BOLD <-> atlas alignment ──────────────────────
report = check_alignment(BOLD_PATH, atlas_img, t_r=TR)

if not report['space_label']['pass']:
    raise RuntimeError(
        'Space label mismatch: BOLD and atlas are in different coordinate spaces. '
        'Verify that your fMRIPrep output is in MNI152 space.'
    )

if not report['field_of_view']['pass']:
    print('WARNING: some atlas parcels fall outside the BOLD field of view.\n'
          'These parcels will produce NaN time series.')

display = plot_alignment(BOLD_PATH, atlas_img, n_cuts=6, display_mode='z')

### AK-5. Extract Task Time Series

In [ ]:
# ── ANSWER KEY: Step 8 — Extract task parcel time series ─────────────────────
# standardize=False and low_pass=None are the two critical differences
# from the resting-state FC pipeline.
time_series = extract_time_series(
    bold_path       = BOLD_PATH,
    atlas_img       = atlas_img,
    confounds_array = confounds_array,
    t_r             = TR,
    standardize     = STANDARDIZE,    # False
    detrend         = True,
    low_pass        = LOWPASS_HZ,     # None
    high_pass       = HIGHPASS_HZ,    # 0.01
)
print(f'Time series shape: {time_series.shape}  (T x ROIs)')

fig = plot_time_series(
    time_series,
    t_r    = TR,
    n_rois = 8,
    labels = roi_labels,
    title  = f'{SUBJECT} — sample parcel time series (first 8 ROIs, unstandardised)',
)
plt.show()

### AK-6. Load Task Events

In [ ]:
TASK_CONDITIONS   = ['checkerboard_left', 'checkerboard_right']
# ── ANSWER KEY: Step 9 — Load task events ────────────────────────────────────
events_df, conditions_found = load_events(
    events_path = EVENTS_PATH,
    conditions  = TASK_CONDITIONS,   # None = keep all trial_type values
)
print(f'Events loaded   : {len(events_df)}')
print(f'Conditions found: {conditions_found}')
print(events_df.head(8).to_string(index=False))

# store task conditons for later...
if TASK_CONDITIONS is None:
    TASK_CONDITIONS = conditions_found  # if this is the variable used in compute_gppi_matrix

### AK-7. Build HRF Regressors

In [ ]:
# ── ANSWER KEY: Step 10 — Build HRF-convolved regressors ─────────────────────
hrf_regressors = build_hrf_regressors(
    events_df  = events_df,
    t_r        = TR,
    n_scans    = time_series.shape[0],
    conditions = TASK_CONDITIONS,
    hrf_model  = 'spm',
)
print(f'Conditions modelled : {list(hrf_regressors.keys())}')
print(f'Regressor shape     : {next(iter(hrf_regressors.values())).shape}')

fig = plot_hrf_regressors(
    hrf_regressors,
    t_r   = TR,
    title = f'{SUBJECT} — HRF-convolved task regressors | {TASK}',
)
plt.show()

### AK-8. Inspect One ROI's Design Matrix (ROI-to-ROI)


In [ ]:
# ── ANSWER KEY: Step 11 — Inspect design matrix for one example ROI ─────────
# Any index works; roi_idx=0 gives the first parcel in the atlas.
# In the full ROI-to-ROI computation (Section 12) this design is rebuilt
# automatically for every ROI, so this step is diagnostic only.
roi_idx = 0
roi_ts  = time_series[:, roi_idx]

design_matrix, col_names, ppi_cols = build_gppi_design(
    seed_ts        = roi_ts,
    hrf_regressors = hrf_regressors,
    confounds      = confounds_array,
)
print(f'Design matrix shape : {design_matrix.shape}')
print(f'Column names        : {col_names}')
print(f'PPI column indices  : {ppi_cols}')
print(f'Example ROI         : {roi_labels[roi_idx]}')

fig = plot_gppi_design_matrix(
    design_matrix,
    col_names,
    title = f'{SUBJECT} — gPPI design matrix | ROI[{roi_idx}]: {roi_labels[roi_idx]}',
)
plt.show()


### AK-9. Compute gPPI Matrix

In [ ]:
# ── ANSWER KEY: Step 12 — Compute gPPI connectivity matrix ───────────────────
gppi_matrix = compute_gppi_matrix(
    time_series = time_series,
    events_df   = events_df,
    t_r         = TR,
    confounds   = confounds_array,
    conditions  = TASK_CONDITIONS,
    hrf_model   = 'spm'
)
print(f'gPPI matrix shape: {gppi_matrix.shape}')
    
for idx, COND in enumerate(TASK_CONDITIONS):
    stats = summarise_gppi(gppi_matrix[:,:,idx], label=f'{SUBJECT} | {ATLAS}-{N_ROIS} | {TASK} | {COND}')

# Optional: symmetrise before group analysis
# gppi_sym = (gppi_matrix + gppi_matrix.T) / 2

### AK-10. Visualise the gPPI Matrix

In [ ]:
# ── ANSWER KEY: Step 13a — Plot gPPI heatmap ─────────────────────────────────
for idx, TASK in enumerate(TASK_CONDITIONS):
    fig = plot_fc_matrix(
        gppi_matrix[:,:,idx],
        labels  = roi_labels,
        title   = f'{SUBJECT} — {TASK.capitalize()} gPPI | '
                  f'{ATLAS.capitalize()}-{N_ROIS} (beta weights)',
        cmap    = 'RdBu_r',
        figsize = (10, 9),
        output_path = os.path.join(OUTPUT_DIR, f'{SUBJECT}_gppi_matrix.png'),
    )
    plt.show()

### AK-11. Plot ROI Connection Profile (post-hoc)


In [ ]:
# ── ANSWER KEY: Step 13b — Profile any ROI from the completed matrix ─────────
# Pick any ROI of interest from roi_labels.  Because the full ROI-to-ROI matrix
# has already been computed, you can explore different ROIs without re-running
# the pipeline.  Try several to understand the spatial pattern of task-modulated
# connectivity.
PROFILE_ROI = roi_labels[0]   # change to any label, e.g. roi_labels[99]

for idx, COND in enumerate(TASK_CONDITIONS):
    fig = plot_gppi_seed_profile(
        gppi_matrix = gppi_matrix[:,:,idx],
        labels      = roi_labels,
        seed_label  = PROFILE_ROI,
        n_top       = 20,
    )
    plt.show()

    # Save
    roi_tag = PROFILE_ROI.replace('/', '_')
    fig.savefig(os.path.join(OUTPUT_DIR,
                f'{SUBJECT}_gppi_cond-{TASK}_roi-{roi_tag}_profile.png'),
                dpi=150, bbox_inches='tight')
    print(f'Profiled: {PROFILE_ROI}')


### AK-12. Save gPPI Matrix

In [ ]:
# ── ANSWER KEY: Step 13c — Save ──────────────────────────────────────────────
for idx, COND in enumerate(TASK_CONDITIONS):
    fname = f'{SUBJECT}_task-{TASK}_atlas-{ATLAS}{N_ROIS}_cond-{COND}_gppi.csv'
    if SESSION:
        fname = f'{SUBJECT}_{SESSION}_task-{TASK}_atlas-{ATLAS}{N_ROIS}_cond-{COND}_gppi.csv'

    out_path = save_fc_matrix(
        fc_matrix  = gppi_matrix[:,:,idx],
        labels     = roi_labels,
        output_dir = OUTPUT_DIR,
        filename   = fname,
    )

# Verify round-trip
check = pd.read_csv(out_path, index_col=0)
print(f'Verified CSV shape: {check.shape}')
print(f'Column names match: {list(check.columns[:3])} ...')

### AK-13. Complete ROI-to-ROI Workflow (single cell)

The entire gPPI pipeline in one compact block — useful as a loop template for multi-subject analysis.


In [ ]:
# ── ANSWER KEY: full gPPI pipeline in one cell ────────────────────────────────
import os
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from fc_helpers import (
    load_confounds, get_parcellation, extract_time_series,
    load_events, build_hrf_regressors, compute_gppi_matrix,
    save_fc_matrix, plot_fc_matrix, plot_gppi_seed_profile, summarise_gppi,
)


def run_gppi_pipeline(
    bold_path, confounds_path, events_path,
    atlas, n_rois, yeo_networks, atlas_path,
    confound_cols, scrub_threshold,
    t_r, output_dir, subject, session=None, task='task',
    task_conditions=None, hrf_model='spm',
    high_pass=0.01, profile_roi_idx=0,
    verbose=True,
):
    """
    Run the complete gPPI pipeline for a single subject/run.

    Returns
    -------
    gppi_matrix : np.ndarray  — shape (n_rois, n_rois), gPPI beta weights
    roi_labels  : list of str
    out_path    : str          — path to saved CSV
    """
    # 1. Parcellation (load once; reuse across subjects)
    atlas_img, roi_labels = get_parcellation(
        atlas=atlas, n_rois=n_rois,
        yeo_networks=yeo_networks, atlas_path=atlas_path,
    )

    # 2. Confounds
    confounds_array, _ = load_confounds(
        confounds_path, confound_cols,
        scrub_threshold=scrub_threshold,
    )

    # 3. Extract task time series  (no low-pass, no standardise)
    ts = extract_time_series(
        bold_path, atlas_img, confounds_array, t_r,
        standardize=False,
        detrend=True,
        low_pass=None,
        high_pass=high_pass,
    )

    # 4. Load events
    events_df, _ = load_events(events_path, conditions=task_conditions)

    # 5. Compute gPPI
    gppi = compute_gppi_matrix(
        ts, events_df, t_r,
        confounds=confounds_array,
        conditions=task_conditions,
        hrf_model=hrf_model,
    )
    print(task_conditions)
    # 6. Save
    sess_tag = f'_{session}' if session else ''
    for idx, COND in enumerate(task_conditions):
        fname    = f'{subject}{sess_tag}_task-{task}_atlas-{atlas}{n_rois}_cond-{COND}_gppi.csv'
        out_path = save_fc_matrix(gppi[:,:,idx], roi_labels, output_dir, filename=fname)

        if verbose:
            summarise_gppi(gppi[:,:,idx], label=f'{subject} | {task} | {COND}')

    return gppi, roi_labels, out_path

# ── Run for the current subject ───────────────────────────────────────────────
gppi_matrix, roi_labels, out_path = run_gppi_pipeline(
    bold_path        = BOLD_PATH,
    confounds_path   = CONFOUNDS_PATH,
    events_path      = EVENTS_PATH,
    atlas            = ATLAS,
    n_rois           = N_ROIS,
    yeo_networks     = YEO_NETWORKS,
    atlas_path       = ATLAS_PATH,
    confound_cols    = CONFOUND_COLS,
    scrub_threshold  = SCRUB_THRESHOLD,
    t_r              = TR,
    output_dir       = OUTPUT_DIR,
    subject          = SUBJECT,
    session          = SESSION,
    task             = TASK,
    task_conditions  = TASK_CONDITIONS,
)

for idx, COND in enumerate(TASK_CONDITIONS):
    fig = plot_fc_matrix(
        gppi_matrix[:,:,idx], labels=roi_labels,
        title=f'{SUBJECT} — {TASK} gPPI | {ATLAS.capitalize()}-{N_ROIS} | {COND}',
    )
    plt.show()

print(f'\nSaved: {out_path}')

### AK-14. Multi-Subject Loop Template

Example of how to scale the gPPI pipeline across subjects and sessions.

In [ ]:
# ── ANSWER KEY: multi-subject loop template ───────────────────────────────────
# Load parcellation once — it is identical across subjects
atlas_img, roi_labels = get_parcellation(
    atlas=ATLAS, n_rois=N_ROIS, yeo_networks=YEO_NETWORKS
)

SUBJECTS = [f'sub-MSC{i:02d}' for i in range(1, 11)]   # sub-MSC01 … sub-MSC10
SESSIONS = [f'ses-func{i:02d}' for i in range(1, 11)]

all_gppi = []   # will hold one (n_rois, n_rois) array per subject-session

for sub in SUBJECTS:
    for ses in SESSIONS:
        # Build paths for this sub/ses
        def fpath(suffix, ext, sub=sub, ses=ses):
            parts  = [sub, ses, 'func']
            tokens = [sub, ses, f'task-{TASK}', suffix]
            return os.path.join(FMRIPREP_DIR, *parts, '_'.join(tokens) + ext)

        bold_path      = fpath(f'space-{SPACE}_res-2_desc-preproc_bold', '.nii.gz')
        confounds_path = fpath('desc-confounds_timeseries', '.tsv')
        events_path    = fpath('events', '.tsv').replace(
            os.path.join(FMRIPREP_DIR, sub, ses, 'func'),
            # Events are typically in the raw BIDS tree, not fMRIPrep — adjust as needed
            os.path.join(FMRIPREP_DIR, sub, ses, 'func'),
        )

        # Skip missing files gracefully
        if not all(os.path.exists(p) for p in [bold_path, confounds_path, events_path]):
            print(f'  SKIP {sub} {ses} — file(s) not found')
            continue

        print(f'\nProcessing {sub} | {ses}')
        conf, _     = load_confounds(confounds_path, CONFOUND_COLS, SCRUB_THRESHOLD)
        ts          = extract_time_series(bold_path, atlas_img, conf, TR,
                                          standardize=False, low_pass=None,
                                          high_pass=HIGHPASS_HZ)
        events, _   = load_events(events_path, TASK_CONDITIONS)
        gppi        = compute_gppi_matrix(ts, events, TR, confounds=conf,
                                          conditions=TASK_CONDITIONS)

        fname = f'{sub}_{ses}_task-{TASK}_atlas-{ATLAS}{N_ROIS}_gppi.csv'
        save_fc_matrix(gppi, roi_labels, OUTPUT_DIR, filename=fname)
        all_gppi.append(gppi)

# Group average gPPI matrix
if all_gppi:
    group_gppi = np.stack(all_gppi, axis=0).mean(axis=0)
    print(f'\nGroup gPPI matrix (n={len(all_gppi)}): {group_gppi.shape}')
    save_fc_matrix(
        group_gppi, roi_labels, OUTPUT_DIR,
        filename=f'group_task-{TASK}_atlas-{ATLAS}{N_ROIS}_gppi.csv',
    )
    fig = plot_fc_matrix(
        group_gppi, labels=roi_labels,
        title=f'Group mean gPPI (n={len(all_gppi)}) | {ATLAS.capitalize()}-{N_ROIS}',
    )
    plt.show()

---
## Summary

| Step | Helper function | Key output |
|------|-----------------|------------|
| Motion QC | `plot_fd_trace()` | FD plot, mean FD, % scrubbed |
| Confounds | `load_confounds()` | `confounds_array` (T × regressors) |
| Parcellation | `get_parcellation()` | `atlas_img`, `roi_labels` |
| Time series | `extract_time_series()` | `time_series` (T × n_rois, **unstandardised, no low-pass**) |
| Events | `load_events()` | `events_df` (onset, duration, trial_type) |
| HRF regressors | `build_hrf_regressors()` | `hrf_regressors` (dict: condition → array) |
| Design matrix | `build_gppi_design()` | `design_matrix`, `col_names`, `ppi_cols` |
| gPPI matrix | `compute_gppi_matrix()` | `gppi_matrix` (n_rois × n_rois, beta weights) |
| Save | `save_fc_matrix()` | `<subject>_gppi.csv` |
| Visualise matrix | `plot_fc_matrix()` | Heatmap figure |
| Seed profile | `plot_gppi_seed_profile()` | Bar chart of top connections |

**Next steps:** Loop over subjects → stack `all_gppi` → group-level GLM to test whether condition-specific connectivity differs between groups, or correlates with behaviour.